# Vault Mechanism Mathematical Modeling

## Core Question
With different Creator equity ratios (10%/30%/50%), after a **+20% gain** followed by a **-16.667% loss** returning to the original capital level, how much does the Creator's capital differ between normal trading vs. vault trading in this "zero-sum cycle"?

## Key Mechanism
- **Losses**: Shared proportionally by share ratio (fair distribution)
- **Profits**: Creator earns an additional performance fee from depositors' profits
- This asymmetry is the Creator's **structural advantage**

In [ ]:
import pandas as pd
import numpy as np

def simulate_vault_cycles(creator_ratio, fee_rate, gain=0.20, loss_rate=1/6, num_cycles=10):
    """
    Simulate vault profit/loss cycles
    
    Parameters:
    - creator_ratio: Creator's equity as a proportion of total vault capital (e.g., 0.1 = 10%)
    - fee_rate: Performance fee rate (e.g., 0.1 = 10%)
    - gain: Profit magnitude per cycle (default 20%)
    - loss_rate: Loss magnitude per cycle (default 16.667%, i.e., 1/6, so that 1.2 * (1-1/6) = 1)
    - num_cycles: Number of profit/loss cycles
    
    Returns: DataFrame with Creator's capital multiplier after each cycle
    """
    # Initial setup
    V = 1000  # Total vault capital
    creator_capital = V * creator_ratio
    initial_creator = creator_capital
    
    # Normal trader (no vault)
    normal_capital = initial_creator
    
    results = []
    
    for cycle in range(1, num_cycles + 1):
        total_vault = creator_capital / creator_ratio if creator_ratio < 1 else creator_capital
        # We actually need to track depositor capital separately
        # Re-modeling: track creator_value and total_vault
        pass
    
    # More rigorous modeling
    creator_val = initial_creator
    depositor_val = initial_creator * (1 - creator_ratio) / creator_ratio
    total = creator_val + depositor_val
    normal = initial_creator
    
    results = [{'cycle': 0, 'creator_val': creator_val, 'normal_val': normal,
                'creator_multiplier': 1.0, 'normal_multiplier': 1.0,
                'advantage': 0.0}]
    
    for cycle in range(1, num_cycles + 1):
        total = creator_val + depositor_val
        
        # Step 1: +20% gain
        profit = total * gain
        creator_profit_share = (creator_val / total) * profit  # Creator's proportional profit
        depositor_profit_share = (depositor_val / total) * profit  # Depositor's proportional profit
        
        # Performance fee: Creator takes a cut from depositor profits
        perf_fee = fee_rate * depositor_profit_share
        
        creator_val_after_gain = creator_val + creator_profit_share + perf_fee
        depositor_val_after_gain = depositor_val + depositor_profit_share - perf_fee
        total_after_gain = creator_val_after_gain + depositor_val_after_gain
        
        # Step 2: -16.667% loss (shared proportionally by shares)
        loss = total_after_gain * loss_rate
        creator_share_ratio = creator_val_after_gain / total_after_gain
        
        creator_val = creator_val_after_gain * (1 - loss_rate)
        depositor_val = depositor_val_after_gain * (1 - loss_rate)
        
        # Normal trader: +20% then -16.667% = back to square one
        normal = normal * (1 + gain) * (1 - loss_rate)
        
        results.append({
            'cycle': cycle,
            'creator_val': creator_val,
            'normal_val': normal,
            'creator_multiplier': creator_val / initial_creator,
            'normal_multiplier': normal / initial_creator,
            'advantage': (creator_val / initial_creator - 1) * 100
        })
    
    return pd.DataFrame(results)

# Verify: +20% then -16.667% is zero-sum for normal traders
print("Verification: 1.2 × (1 - 1/6) =", 1.2 * (1 - 1/6))
print()

## Scenario 1: Hyperliquid (Fixed 10% Performance Fee)

In [ ]:
# ============================================================
# Hyperliquid: Fixed 10% Performance Fee
# Creator Equity Ratio: 10%, 30%, 50%
# ============================================================

print("=" * 80)
print("Hyperliquid Vault (Performance Fee = 10%)")
print("=" * 80)

for ratio in [0.10, 0.30, 0.50]:
    df = simulate_vault_cycles(creator_ratio=ratio, fee_rate=0.10, num_cycles=10)
    print(f"\n▶ Creator Equity Ratio: {ratio*100:.0f}%")
    print(f"{'Cycle':>6} | {'Creator Mult.':>14} | {'Normal Mult.':>14} | {'Advantage':>10}")
    print("-" * 55)
    for _, row in df.iterrows():
        if row['cycle'] in [0, 1, 5, 10]:
            print(f"{int(row['cycle']):>6} | {row['creator_multiplier']:>14.4f} | {row['normal_multiplier']:>14.4f} | {row['advantage']:>9.2f}%")
    print()

## Scenario 2: DipCoin (Customizable Performance Fee 1%-50%)

In [ ]:
# ============================================================
# DipCoin: Customizable Performance Fee (showing 10%, 20%, 30%, 50%)
# Creator Equity Ratio: 10%, 30%, 50%
# ============================================================

print("=" * 80)
print("DipCoin Vault - Performance Fee Comparison")
print("=" * 80)

for fee in [0.10, 0.20, 0.30, 0.50]:
    print(f"\n{'='*60}")
    print(f"Performance Fee = {fee*100:.0f}%")
    print(f"{'='*60}")
    for ratio in [0.10, 0.30, 0.50]:
        df = simulate_vault_cycles(creator_ratio=ratio, fee_rate=fee, num_cycles=10)
        row_1 = df[df['cycle'] == 1].iloc[0]
        row_5 = df[df['cycle'] == 5].iloc[0]
        row_10 = df[df['cycle'] == 10].iloc[0]
        print(f"  Equity {ratio*100:>2.0f}% | 1 cycle: {row_1['creator_multiplier']:.4f}x | 5 cycles: {row_5['creator_multiplier']:.4f}x | 10 cycles: {row_10['creator_multiplier']:.4f}x")

## Comprehensive Comparison Table (for article)

In [ ]:
# ============================================================
# Generate comprehensive comparison table for article
# ============================================================

print("Comprehensive Comparison: Creator Capital Multiplier After N '+20% → -16.667%' Cycles")
print("(Normal trader always remains at 1.0000x, since each cycle has zero net return)")
print()

header = f"{'Perf Fee':>8} | {'Equity %':>8} | {'1 Cycle':>8} | {'5 Cycles':>9} | {'10 Cycles':>10}"
print(header)
print("-" * len(header))

for fee in [0.10, 0.20, 0.30, 0.50]:
    for ratio in [0.10, 0.30, 0.50]:
        df = simulate_vault_cycles(creator_ratio=ratio, fee_rate=fee, num_cycles=10)
        r1 = df[df['cycle']==1].iloc[0]['creator_multiplier']
        r5 = df[df['cycle']==5].iloc[0]['creator_multiplier']
        r10 = df[df['cycle']==10].iloc[0]['creator_multiplier']
        print(f"{fee*100:>7.0f}% | {ratio*100:>7.0f}% | {r1:>8.4f} | {r5:>9.4f} | {r10:>10.4f}")
    print("-" * len(header))

## Expected Value Calculation

Assume the probability of profit per trade is p, and the probability of loss is (1-p).

In [ ]:
# ============================================================
# Expected Value Analysis
# ============================================================

def expected_multiplier_per_trade(p, fee_rate, creator_ratio, gain=0.20, loss_rate=1/6):
    """
    Calculate expected return multiplier per single trade
    
    Normal trader:
    E_normal = p × (1 + gain) + (1-p) × (1 - loss_rate)
    
    Vault Creator:
    Win multiplier = (1 + gain) + fee × (1-X)/X × gain
    Loss multiplier = (1 - loss_rate)   (same as normal, proportional loss)
    
    E_vault = p × [(1+gain) + fee×(1-X)/X×gain] + (1-p) × (1-loss_rate)
    
    Creator's expected advantage:
    E_vault - E_normal = p × fee × (1-X)/X × gain
    """
    X = creator_ratio
    
    # Creator's win multiplier
    win_mult_vault = (1 + gain) + fee_rate * (1 - X) / X * gain
    win_mult_normal = (1 + gain)
    
    # Loss multiplier (same for both)
    lose_mult = (1 - loss_rate)
    
    E_vault = p * win_mult_vault + (1 - p) * lose_mult
    E_normal = p * win_mult_normal + (1 - p) * lose_mult
    
    return E_vault, E_normal, E_vault - E_normal

print("Expected Value Analysis (Per Single Trade)")
print("=" * 70)
print()
print("Formula Derivation:")
print("  Normal Trader  E = p × 1.2 + (1-p) × 0.8333")
print("  Vault Creator  E = p × [1.2 + fee × (1-X)/X × 0.2] + (1-p) × 0.8333")
print("  Creator Edge  ΔE = p × fee × (1-X)/X × 0.2")
print()

print(f"{'Win%':>5} | {'Fee':>6} | {'Equity X':>8} | {'E_vault':>8} | {'E_normal':>8} | {'Edge ΔE':>8}")
print("-" * 60)

for p in [0.45, 0.50, 0.55]:
    for fee in [0.10, 0.30, 0.50]:
        for X in [0.10, 0.30]:
            ev, en, delta = expected_multiplier_per_trade(p, fee, X)
            print(f"{p:>5.2f} | {fee*100:>5.0f}% | {X*100:>7.0f}% | {ev:>8.4f} | {en:>8.4f} | {delta:>+8.4f}")
    print("-" * 60)

print()
print("Key Finding:")
print("Even with a win rate of only 45% (below 50%), Vault Creators still gain a positive edge via the fee mechanism!")
print("Advantage formula: ΔE = p × fee × (1-X)/X × gain")

## Adjustable Parameters - Try It Yourself

In [ ]:
# ============================================================
# 🔧 Adjustable Parameters - Modify these to test different scenarios
# ============================================================

# Parameter settings
MY_CREATOR_RATIO = 0.10    # Creator equity ratio (0.01 ~ 0.99)
MY_FEE_RATE = 0.30         # Performance fee rate (0.01 ~ 0.50)
MY_GAIN = 0.20             # Profit magnitude
MY_LOSS = 1/6              # Loss magnitude (1/6 ≈ 16.667%)
MY_CYCLES = 20             # Number of cycles

# Run simulation
df = simulate_vault_cycles(
    creator_ratio=MY_CREATOR_RATIO,
    fee_rate=MY_FEE_RATE,
    gain=MY_GAIN,
    loss_rate=MY_LOSS,
    num_cycles=MY_CYCLES
)

print(f"Parameters: Equity={MY_CREATOR_RATIO*100:.0f}%, Fee={MY_FEE_RATE*100:.0f}%, "
      f"Gain={MY_GAIN*100:.0f}%, Loss={MY_LOSS*100:.2f}%")
print()
print(df[['cycle', 'creator_multiplier', 'normal_multiplier', 'advantage']].to_string(index=False))